1. Install & Imports

In [1]:
import pandas as pd
import re
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

2. Load Dataset

In [2]:
dataset = load_dataset("code_search_net", "python")
print(dataset)

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

C:\Users\chand\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chand\.cache\huggingface\hub\datasets--code_search_net. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})


3. Use Small Subset

In [3]:
train_data = dataset['train'].select(range(10000))
test_data = dataset['test'].select(range(2000))

4. Clean Code Function

In [4]:
def clean_code(code):
    code = re.sub(r"#.*", "", code)  # remove comments
    code = re.sub(r"\n\s*\n", "\n", code)  # remove empty lines
    return code.strip()

5. Create Labels (SMART LOGIC)

In [5]:
def create_label(code):
    # heuristic rules
    if code.count("for") > 2:
        return 1  # complex / inefficient
    if "while True" in code:
        return 1  # risky
    if len(code) > 500:
        return 1
    return 0

6. Apply Processing

In [6]:
def process_dataset(ds):
    codes = []
    labels = []

    for item in tqdm(ds):
        code = clean_code(item["whole_func_string"])
        label = create_label(code)

        codes.append(code)
        labels.append(label)

    return pd.DataFrame({"code": codes, "label": labels})

train_df = process_dataset(train_data)
test_df = process_dataset(test_data)

100%|████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:01<00:00, 1172.91it/s]


7. Tokenization

In [7]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")

def tokenize_function(examples):
    return tokenizer(
        examples["code"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

C:\Users\chand\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chand\.cache\huggingface\hub\models--microsoft--codebert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

8. Convert to Dataset Format

In [8]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

9. Save Processed Data

In [9]:
train_dataset.save_to_disk("../data/processed/train")
test_dataset.save_to_disk("../data/processed/test")

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]